## 11. 🆕 Sensibilidad (ρ, τ, N), hipótesis testeables y control

Esta sección **depende solo de variables ya en memoria** del notebook principal: `d` (cohortes 1950–2035), `s` (series por año), `sens` (sensibilidad de $\bar A$, sección 7), `TAU`, `RHO`, `A_OBS`, `A_LEGAL`, `DATA` y `OUT`. Escribe tablas y gráficos en `OUT/salidas/`.

**Qué es y qué no es.** El TBP es una **identidad determinística**: no hay variable resultado ni error aleatorio. Por eso **no corresponde un test de significancia**. La validación se hace por (i) **estática comparativa** (mover un parámetro y mirar cuánto cambia el índice), (ii) **escenarios** y (iii) **comparación con un indicador convencional** (razón de dependencia). No se afirma novedad.

Recordatorio algebraico: $TBP_C = \dfrac{N(1-\mu)\,\bar A\,\tau}{J\,\rho}$ (años), $TBP_A = TBP_C / E_r$, $TBP_A = D\times B$ con $D = N(1-\mu)/J$ y $B = \bar A\tau/(E_r\rho)$.

In [ ]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt

SAL = os.path.join(OUT, "salidas")          # CSV y PNG de esta seccion (NO datos/procesados)
os.makedirs(SAL, exist_ok=True)

# ---- Parametros con fuente: Anuario Estadistico ANSES 2008-2023 (datos_tbp/Anuario Estadístico_final.pdf) ----
#  p.91: tasa de sustitucion = haber medio SIPA (dic) / salario imponible medio (nov)
#  p.39: aporte personal 11%; contribucion patronal SIPA 10,77% (inc. a), 12,35% (inc. b), 16% (admin. publica)
ANSES = dict(rho_dic2009=0.371, rho_dic2023=0.403,
             rho_sin_moratoria_2009=0.425, rho_sin_moratoria_2023=0.563,
             aporte_personal=0.11, contrib_inc_a=0.1077, contrib_inc_b=0.1235, contrib_admin_publica=0.16)
TAU_A  = ANSES["aporte_personal"] + ANSES["contrib_inc_a"]          # 21,77 %
TAU_B  = ANSES["aporte_personal"] + ANSES["contrib_inc_b"]          # 23,35 %
TAU_AP = ANSES["aporte_personal"] + ANSES["contrib_admin_publica"]  # 27 %
RHO_09, RHO_23, RHO_SM = ANSES["rho_dic2009"], ANSES["rho_dic2023"], ANSES["rho_sin_moratoria_2023"]

# ---- Control 4c: los valores citados coinciden con los del notebook y con lo documentado ----
assert np.isclose(TAU, 0.2177) and np.isclose(TAU, TAU_A)
assert np.isclose(TAU_B, 0.2335) and np.isclose(TAU_AP, 0.27)
assert np.isclose(RHO, 0.403) and np.isclose(RHO, RHO_23)
assert np.isclose(RHO_09, 0.371) and np.isclose(RHO_SM, 0.563)
assert A_OBS == 14.2 and A_LEGAL == 30
print("OK 4c) tau=21,77% (23,35% / 27% alternativos), rho=40,3% (37,1% / 56,3% alternativos), A=14,2 y 30: coinciden con lo citado")

c = d.set_index("cohorte")
def tbp_C(N, S, J, A, tau, rho):            return N * S * A * tau / (J * rho)
def tbp_A(N, S, J, E, A, tau, rho):         return N * S * A * tau / (J * E * rho)

# consistencia con el notebook principal (CSV y seccion 7)
assert np.allclose(tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, RHO), c.TBP_C_legal)
assert np.allclose(tbp_A(c.N_t, c.S65, c.J, c.E65, A_OBS, TAU, RHO), c.TBP_A_obs)
_chk = sens[(sens.A_barra == 30) & (sens.cohorte == 2024)].TBP_C.iloc[0]
assert np.isclose(_chk, c.TBP_C_legal[2024])
print(f"OK) funciones reproducen TBP_C/TBP_A del notebook principal; TBP_C(2024, A=30) = {c.TBP_C_legal[2024]:.3f} años")

### 11.1 Sensibilidad a $\rho$ y $\tau$ (y $\bar A$, $N$)

**Rangos justificados** (todos con fuente, salvo lo rotulado como ilustración):

* $\rho$: 0,371 (dic-2009), 0,403 (dic-2023, **base**) y 0,563 (haber contributivo **sin moratoria**, 2023) — Anuario ANSES p. 91.
* $\tau$: 0,2177 (11% + 10,77%, **base**), 0,2335 (11% + 12,35%, inciso b) y 0,27 (11% + 16%, administración pública) — Anuario ANSES p. 39.
* $\bar A$: 15 y 30 (30 es un requisito legal de referencia, **no un máximo**; 14,2 es un valor observado sesgado).

**Qué esperar:** $TBP_C$ es **lineal** en $\bar A$, $\tau$ y $1/\rho$ ⇒ estos parámetros mueven el **nivel** de toda la curva por un factor constante, no su forma. Lo que *sí* cambiaría la forma es que un parámetro **dependa de la cohorte** (ejemplo ilustrativo más abajo, **no es un dato**).

In [ ]:
RHOS = [RHO_09, RHO, RHO_SM]; TAUS = [TAU, TAU_B, TAU_AP]; AS = [15.0, 30.0]; COH = [2000, 2010, 2024]
filas = []
for t in COH:
    for A in AS:
        for rho in RHOS:
            for tau in TAUS:
                filas.append(dict(cohorte=t, A_barra=A, rho=rho, tau=tau,
                                  TBP_C=tbp_C(c.N_t[t], c.S65[t], c.J[t], A, tau, rho)))
grid = pd.DataFrame(filas)

# ---- Control 4a: linealidad en rho (TBP*rho constante) y en tau (TBP/tau constante) y en A ----
for (t, A), g in grid.groupby(["cohorte", "A_barra"]):
    assert np.allclose(g.TBP_C * g.rho / g.tau, g.TBP_C.iloc[0] * g.rho.iloc[0] / g.tau.iloc[0])   # TBP*rho/tau = const
for rho in RHOS:
    for tau in TAUS:
        g = grid[(grid.rho == rho) & (grid.tau == tau)].pivot(index="cohorte", columns="A_barra", values="TBP_C")
        assert np.allclose(g[30.0], 2 * g[15.0])                       # lineal en A
base_ = lambda t: tbp_C(c.N_t[t], c.S65[t], c.J[t], A_LEGAL, TAU, RHO)
for t in COH:
    for k in (0.5, 2.0):                                                # escalar tau o 1/rho por k escala TBP por k
        assert np.isclose(tbp_C(c.N_t[t], c.S65[t], c.J[t], A_LEGAL, k * TAU, RHO), k * base_(t))
        assert np.isclose(tbp_C(c.N_t[t], c.S65[t], c.J[t], A_LEGAL, TAU, RHO / k), k * base_(t))
print("OK 4a) linealidad: TBP_C*rho/tau constante, TBP_C(k*tau)=k*TBP_C, TBP_C(rho/k)=k*TBP_C, TBP_C(2A)=2*TBP_C")

tabla = grid.pivot_table(index=["A_barra", "cohorte", "rho"], columns="tau", values="TBP_C")
tabla.columns = [f"tau={x:.4f}" for x in tabla.columns]
print("TBP_C (años) por cohorte x Ā x ρ x τ  (tau=0,2177 base; 0,2335 inc. b; 0,27 admin. pública):")
display(tabla.round(3))

rng = grid.groupby("cohorte").TBP_C.agg(["min", "max"]); rng["max/min"] = rng["max"] / rng["min"]
print("\nRango de TBP_C en toda la grilla (3 ρ × 3 τ × 2 Ā):"); display(rng.round(3))
print("Solo ρ (τ y Ā base, Ā=30): TBP_C(ρ=0,563)/TBP_C(ρ=0,371) =", round(RHO_09 / RHO_SM, 3), "-> factor constante en todas las cohortes")
tabla.reset_index().to_csv(os.path.join(SAL, "s11_grilla_rho_tau_A.csv"), index=False)

In [ ]:
# ---- Curvas por cohorte para 3 valores de rho (Ā=30, tau base) + ejemplo ILUSTRATIVO no lineal ----
coh = c.index.values
fig, ax = plt.subplots(1, 3, figsize=(19, 5.2))
for rho, col, lab in zip(RHOS, ["tab:green", "tab:blue", "tab:red"],
                         ["ρ=0,371 (dic-2009)", "ρ=0,403 (dic-2023, base)", "ρ=0,563 (sin moratoria 2023)"]):
    ax[0].plot(coh, tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, rho), color=col, label=lab)
ax[0].set_title("TBP_C por cohorte según ρ (Ā=30, τ=0,2177)\nρ constante ⇒ cambia el NIVEL, no la forma", fontsize=10)

# ILUSTRATIVO: rho cae para cohortes jovenes por menor cobertura. NO ES UN DATO.
def rho_ilus(t):
    return RHO * (1 - 0.40 * np.clip((np.asarray(t, dtype=float) - 1975) / (2000 - 1975), 0, 1))
tb_const = tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, RHO)
tb_ilus  = tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, rho_ilus(coh))
ax[1].plot(coh, tb_const, color="tab:blue", label="ρ constante = 0,403")
ax[1].plot(coh, tb_ilus, color="tab:orange", ls="--", label="ρ(cohorte) ILUSTRATIVO")
ax[1].set_title("EJEMPLO ILUSTRATIVO — NO ES DATO: ρ cae 40% entre\ncohortes 1975 y 2000 (supuesto inventado, menor cobertura)", fontsize=10, color="firebrick")
idx_c = 100 * tb_const / tb_const[1950]; idx_i = 100 * tb_ilus / tb_ilus[1950]
ax[2].plot(coh, idx_c, color="tab:blue", label="ρ constante (índice 1950=100)")
ax[2].plot(coh, idx_i, color="tab:orange", ls="--", label="ρ(cohorte) ILUSTRATIVO")
ax[2].set_title("Índice 1950=100: con ρ(cohorte) la FORMA sí cambia\n(ilustrativo, no es dato)", fontsize=10, color="firebrick")
for a in ax:
    a.axvline(2024.5, color="gray", ls=":"); a.grid(alpha=.3); a.legend(fontsize=8); a.set_xlabel("año de nacimiento de la cohorte")
ax[0].set_ylabel("TBP_C (años)")
plt.tight_layout(); plt.savefig(os.path.join(SAL, "s11_curvas_rho.png"), dpi=130); plt.show()

# Control: con rho constante el indice normalizado es identico entre los 3 rho; con rho(cohorte) NO lo es
idx_rho = [100 * tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, r) / tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, r)[1950] for r in RHOS]
assert all(np.allclose(idx_rho[0], x) for x in idx_rho[1:])
assert np.ptp(idx_i - idx_c) > 5          # la forma cambia de manera apreciable
print(f"OK) índice 2024 (1950=100): rho constante = {idx_c[2024]:.1f} (igual para cualquier rho) | rho(cohorte) ilustrativo = {idx_i[2024]:.1f}")
print("Lectura: TBP menor ρ ⇒ MAYOR TBP (la prestación es más barata respecto del aporte); TBP alto no significa 'mejor' para el jubilado.")

In [ ]:
# ---- Tornado para la cohorte 2024: variacion % de TBP_C al mover cada parametro dentro de su rango ----
t0 = 2024
N0, S0, J0 = c.N_t[t0], c.S65[t0], c.J[t0]
base = tbp_C(N0, S0, J0, A_LEGAL, TAU, RHO)          # base: A=30 (referencia legal), rho=0,403, tau=0,2177
rangos = [   # (nombre, valor bajo, valor alto, funcion TBP(valor))
    ("Ā  [15 – 30]",                         15.0, 30.0,       lambda v: tbp_C(N0, S0, J0, v, TAU, RHO)),
    ("ρ  [0,371 – 0,563]",                   RHO_SM, RHO_09,   lambda v: tbp_C(N0, S0, J0, A_LEGAL, TAU, v)),   # rho alto => TBP bajo
    ("τ  [0,2177 – 0,27]",                   TAU, TAU_AP,      lambda v: tbp_C(N0, S0, J0, A_LEGAL, v, RHO)),
    ("J  ±10% (ILUSTRATIVO, proyección ONU)", 1.10 * J0, 0.90 * J0, lambda v: tbp_C(N0, S0, v, A_LEGAL, TAU, RHO)),
    ("N  ±10% (ILUSTRATIVO)",                0.90 * N0, 1.10 * N0, lambda v: tbp_C(v, S0, J0, A_LEGAL, TAU, RHO)),
]
torn = pd.DataFrame([dict(parametro=n, valor_en_extremo_bajo_TBP=lo, valor_en_extremo_alto_TBP=hi,
                          var_pct_bajo=100 * (f(lo) / base - 1), var_pct_alto=100 * (f(hi) / base - 1))
                     for n, lo, hi, f in rangos])
torn["amplitud_pp"] = torn.var_pct_alto - torn.var_pct_bajo
torn = torn.sort_values("amplitud_pp")
display(torn.round(2))

fig, ax = plt.subplots(figsize=(9, 4.2))
y = np.arange(len(torn))
ax.barh(y, torn.var_pct_bajo, color="tab:red", alpha=.8, label="extremo que baja TBP")
ax.barh(y, torn.var_pct_alto, color="tab:blue", alpha=.8, label="extremo que sube TBP")
ax.set_yticks(y); ax.set_yticklabels(torn.parametro, fontsize=8.5); ax.axvline(0, color="k", lw=.8)
ax.set_xlabel(f"variación % de TBP_C (cohorte 2024)\nbase = {base:.2f} años (Ā=30, ρ=0,403, τ=0,2177)", fontsize=9)
ax.set_title("Tornado: cohorte 2024 (N de DEIS; J y S65 proyección ONU)", fontsize=10); ax.grid(alpha=.3, axis="x"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(SAL, "s11_tornado_cohorte2024.png"), dpi=130); plt.show()
# controles aritmeticos del tornado (lineales / hiperbolica)
assert np.isclose(torn.set_index("parametro").loc["Ā  [15 – 30]", "var_pct_bajo"], -50.0)
assert np.isclose(torn.set_index("parametro").loc["N  ±10% (ILUSTRATIVO)", "var_pct_bajo"], -10.0)
assert np.isclose(torn.set_index("parametro").loc["ρ  [0,371 – 0,563]", "var_pct_bajo"], 100 * (RHO / RHO_SM - 1))
torn.to_csv(os.path.join(SAL, "s11_tornado_cohorte2024.csv"), index=False)
print("Nota: la base usa Ā=30 (referencia legal, no un máximo); con otra base de Ā los porcentajes de Ā cambian, los de ρ, τ y N no.")

### 11.2 Hipótesis testeables y estática comparativa

Cada hipótesis se formula con **signo esperado**, **criterio de rechazo** y **la parte del modelo** que la pone a prueba. **Honestidad metodológica:** como el TBP es una identidad, las "pruebas" que siguen son **comparaciones de estática comparativa** (¿el signo y la magnitud calculados con las cohortes reales coinciden con lo esperado?), **no pruebas estadísticas**: no hay p-valores ni intervalos de confianza, y las cohortes sucesivas no son observaciones independientes.

| | Hipótesis | Signo esperado | Parte del modelo | Criterio de rechazo |
|---|---|---|---|---|
| H1 | Una caída sostenida de $N$ respecto de las cohortes que integran $J$ reduce el TBP | − (vía $D$) | numerador $N$ y denominador $J$ de $D$ | La contribución de $\Delta\ln N$ a $\Delta\ln TBP$ tiene signo contrario, o $N$ cae y TBP no cae con el resto fijo |
| H2 | Mayor longevidad ($E_r$, $J$) reduce el TBP | − (doble canal en $TBP_A$; un canal en $TBP_C$) | $E_r$ y $J$ | La elasticidad de $TBP_A$ a $E_r$ no es −1, o la contribución de $-\Delta\ln J$ no es negativa cuando $J$ crece |
| H3 | Mayor densidad de aportes $\bar A$ aumenta el TBP (**nivel, no forma**) | + | $\bar A$ | Elasticidad ≠ +1, o el índice normalizado cambia con $\bar A$ |
| H4 | Mayor formalidad (más $\bar A$ y/o $\tau$ efectiva) aumenta el TBP | + | $\bar A$, $\tau$ | Igual que H3/H5; **la formalidad por cohorte requiere EPH (sección 8, sin datos reales todavía)** |
| H5 | Mayor tasa de aporte $\tau$ aumenta el TBP | + | $\tau$ | Elasticidad ≠ +1 |
| H6 | Mayor tasa de reemplazo $\rho$ reduce el TBP | − | $\rho$ | Elasticidad ≠ −1 |

Nota: $S_{65}=1-\mu$ (supervivencia hasta los 65) **aumenta** el TBP: más longevidad *antes* de 65 y más longevidad *después* de 65 empujan en sentidos opuestos. Por eso H2 se refiere a la longevidad **posterior** a los 65 ($E_r$) y al stock de mayores $J$.

In [ ]:
# ---- Elasticidades discretas (arco, +10%) en la cohorte 2024: eps = ln(TBP1/TBP0) / ln(x1/x0) ----
t0 = 2024
P0 = dict(N=c.N_t[t0], S=c.S65[t0], J=c.J[t0], E=c.E65[t0], A=A_LEGAL, tau=TAU, rho=RHO)
def f_C(p): return tbp_C(p["N"], p["S"], p["J"], p["A"], p["tau"], p["rho"])
def f_A(p): return tbp_A(p["N"], p["S"], p["J"], p["E"], p["A"], p["tau"], p["rho"])
esp = {"N": (1, 1), "S": (1, 1), "J": (-1, -1), "E": (0, -1), "A": (1, 1), "tau": (1, 1), "rho": (-1, -1)}
et = []
for k in P0:
    p1 = dict(P0); p1[k] = 1.10 * P0[k]
    eC = np.log(f_C(p1) / f_C(P0)) / np.log(1.10); eA = np.log(f_A(p1) / f_A(P0)) / np.log(1.10)
    assert np.isclose(eC, esp[k][0]) and np.isclose(eA, esp[k][1])
    et.append(dict(parametro=k, elasticidad_TBP_C=eC, elasticidad_TBP_A=eA, hipotesis={"N":"H1","S":"-","J":"H1/H2","E":"H2","A":"H3/H4","tau":"H4/H5","rho":"H6"}[k]))
elas = pd.DataFrame(et)
display(elas.round(3))
print("OK) signos y magnitudes de H1–H6 coinciden con lo esperado (elasticidades ±1; E_r: 0 en TBP_C, −1 en TBP_A).")
print("Ojo: estas elasticidades son propiedades del diseño del índice (cada factor entra con exponente ±1), no un resultado empírico.")

In [ ]:
# ---- Descomposición logarítmica EXACTA de la caída del TBP entre cohortes ----
#  ln TBP_C = ln N + ln S65 + ln Ā + ln τ − ln J − ln ρ  =>  con Ā, τ, ρ constantes:
#  Δln TBP_C = Δln N + Δln S65 − Δln J ;   Δln TBP_A = Δln TBP_C − Δln E_r
def descomponer(t0, t1, tipo="C"):
    comp = {"N (natalidad)": np.log(c.N_t[t1] / c.N_t[t0]),
            "S65 (supervivencia a 65)": np.log(c.S65[t1] / c.S65[t0]),
            "J (stock 65+; envejecimiento)": -np.log(c.J[t1] / c.J[t0])}
    if tipo == "A": comp["E_r (longevidad a los 65)"] = -np.log(c.E65[t1] / c.E65[t0])
    col = "TBP_C_legal" if tipo == "C" else "TBP_A_legal"
    total = np.log(c[col][t1] / c[col][t0])           # calculado desde el indice, independiente de los componentes
    assert np.isclose(sum(comp.values()), total, atol=1e-12)                      # control 4b
    sh = {k: 100 * v / total for k, v in comp.items()}
    assert np.isclose(sum(sh.values()), 100.0)
    return comp, total, sh

res = []
for (t0_, t1_) in [(1990, 2024), (2000, 2024), (2010, 2024)]:
    for tipo in "CA":
        comp, total, sh = descomponer(t0_, t1_, tipo)
        row = dict(desde=t0_, hasta=t1_, indice="TBP_" + tipo, dlnTBP=total, variacion_pct=100 * (np.exp(total) - 1))
        for k in comp: row["dln " + k] = comp[k]; row["% " + k] = sh[k]
        row["suma_%"] = sum(sh.values()); res.append(row)
desc = pd.DataFrame(res)
print("OK 4b) la suma de las contribuciones = ΔlnTBP (100%) en todas las comparaciones")
display(desc[["desde", "hasta", "indice", "variacion_pct", "dlnTBP"] + [x for x in desc.columns if x.startswith("% ")] + ["suma_%"]].round(2))

comp, total, sh = descomponer(2000, 2024, "C")
print(f"\nTBP_C cohorte 2000 -> 2024: {c.TBP_C_legal[2000]:.3f} -> {c.TBP_C_legal[2024]:.3f} años ({100*(np.exp(total)-1):.1f}%); Δln = {total:.4f}")
for k in comp: print(f"   {k:34s} Δln = {comp[k]:+.4f}  -> {sh[k]:6.1f}% de la caída")

# Detalle de H1: N de la cohorte vs. promedio de N de las cohortes que integran J (nacidas t-35..t, 65 a 100 años en t+65)
Nref = pd.Series({t: c.N_t.loc[t - 35:t].mean() for t in range(1985, 2036)})
for (t0_, t1_) in [(2000, 2024)]:
    rel = np.log((c.N_t[t1_] / Nref[t1_]) / (c.N_t[t0_] / Nref[t0_])); ref = np.log(Nref[t1_] / Nref[t0_])
    assert np.isclose(rel + ref, np.log(c.N_t[t1_] / c.N_t[t0_]))
    print(f"\nDetalle H1 ({t0_}->{t1_}): Δln N = {comp['N (natalidad)']:+.4f} = {rel:+.4f} (N relativo al promedio de las 36 cohortes que integran J) + {ref:+.4f} (nivel de ese promedio)")
print("Lectura: J agrega natalidad y longevidad de cohortes anteriores; por eso la contribución de J no es 'longevidad pura'.")
desc.to_csv(os.path.join(SAL, "s11_descomposicion_log.csv"), index=False)
elas.to_csv(os.path.join(SAL, "s11_elasticidades_cohorte2024.csv"), index=False)

fig, ax = plt.subplots(figsize=(8, 4))
comps = {"C": descomponer(2000, 2024, "C")[0], "A": descomponer(2000, 2024, "A")[0]}
labels = list(comps["A"].keys()); xx = np.arange(len(labels)); w = .38
ax.bar(xx - w/2, [comps["C"].get(k, 0) for k in labels], w, label="TBP_C (recomendada)", color="tab:blue")
ax.bar(xx + w/2, [comps["A"][k] for k in labels], w, label="TBP_A (con E_r)", color="tab:orange")
ax.axhline(0, color="k", lw=.8); ax.set_xticks(xx); ax.set_xticklabels(labels, fontsize=8, rotation=10)
ax.set_ylabel("contribución a Δln TBP (2000 → 2024)"); ax.set_title("Descomposición logarítmica exacta de la caída 2000→2024", fontsize=10)
ax.grid(alpha=.3, axis="y"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(SAL, "s11_descomposicion_2000_2024.png"), dpi=130); plt.show()

In [ ]:
# ---- Tabla de hipotesis con la evidencia numerica de estatica comparativa ----
cC, tC, shC = descomponer(2000, 2024, "C"); cA, tA, shA = descomponer(2000, 2024, "A")
e = elas.set_index("parametro")
cf_N = tbp_C(c.N_t[2000], c.S65[2024], c.J[2024], A_LEGAL, TAU, RHO) / c.TBP_C_legal[2024] - 1
hip = pd.DataFrame([
 dict(id="H1", hipotesis="Caída sostenida de N respecto de las cohortes que integran J reduce TBP (vía D)", signo="−",
      parte_del_modelo="N y J en D", criterio_rechazo="Contribución de Δln N con signo opuesto, o N cae y TBP_C no cae (resto fijo)",
      evidencia=f"2000→2024: N {c.N_t[2000]:,.0f}→{c.N_t[2024]:,.0f}; contribución de N = {shC['N (natalidad)']:.1f}% de la caída de TBP_C; con N de 2000 y el resto de 2024, TBP_C(2024) sería {100*cf_N:+.0f}% mayor",
      estado="no rechazada (comparación de estática comparativa)"),
 dict(id="H2", hipotesis="Mayor longevidad (E_r y J) reduce TBP (doble canal en TBP_A, uno en TBP_C)", signo="−",
      parte_del_modelo="E_r y J", criterio_rechazo="Elasticidad de TBP_A a E_r ≠ −1 o contribución de J no negativa cuando J crece",
      evidencia=f"elasticidad TBP_A a E_r = {e.loc['E','elasticidad_TBP_A']:.0f}, a J = {e.loc['J','elasticidad_TBP_A']:.0f}; 2000→2024: contrib. de J = {shC['J (stock 65+; envejecimiento)']:.1f}% (TBP_C), de E_r = {shA['E_r (longevidad a los 65)']:.1f}% (TBP_A); S65 empuja en sentido contrario ({shC['S65 (supervivencia a 65)']:.1f}%)",
      estado="no rechazada (E_r casi no cambia; el canal J es el relevante)"),
 dict(id="H3", hipotesis="Mayor densidad de aportes Ā aumenta TBP (nivel, no forma)", signo="+",
      parte_del_modelo="Ā", criterio_rechazo="Elasticidad ≠ +1 o el índice normalizado depende de Ā",
      evidencia=f"elasticidad = {e.loc['A','elasticidad_TBP_C']:.0f}; Ā 15→30 duplica TBP_C; índice 1950=100 idéntico (sección 7)", estado="no rechazada (propiedad del diseño)"),
 dict(id="H4", hipotesis="Mayor formalidad = mayor Ā y/o τ efectiva ⇒ TBP mayor", signo="+",
      parte_del_modelo="Ā y τ", criterio_rechazo="Como H3/H5; requiere Ā por cohorte (EPH, sección 8)",
      evidencia=f"efecto de signo conocido (+); ejemplo: Ā 15→22,5 (+50%) y τ 0,2177→0,27 (+24%) ⇒ TBP_C ×{1.5*TAU_AP/TAU:.2f}", estado="signo verificado; magnitud NO evaluable sin datos EPH"),
 dict(id="H5", hipotesis="Mayor τ aumenta TBP", signo="+", parte_del_modelo="τ", criterio_rechazo="Elasticidad ≠ +1",
      evidencia=f"elasticidad = {e.loc['tau','elasticidad_TBP_C']:.0f}; τ 0,2177→0,27: {100*(TAU_AP/TAU-1):+.1f}%", estado="no rechazada (propiedad del diseño)"),
 dict(id="H6", hipotesis="Mayor ρ reduce TBP", signo="−", parte_del_modelo="ρ", criterio_rechazo="Elasticidad ≠ −1",
      evidencia=f"elasticidad = {e.loc['rho','elasticidad_TBP_C']:.0f}; ρ 0,403→0,563: {100*(RHO/RHO_SM-1):+.1f}%; 0,403→0,371: {100*(RHO/RHO_09-1):+.1f}%", estado="no rechazada (propiedad del diseño)"),
])
pd.set_option("display.max_colwidth", 200)
display(hip[["id", "signo", "parte_del_modelo", "evidencia", "estado"]])
hip.to_csv(os.path.join(SAL, "s11_hipotesis.csv"), index=False)
print("Aclaración: 'no rechazada' = el signo y la magnitud calculados coinciden con lo esperado; NO es un resultado estadístico.")

### 11.3 Control con un indicador convencional: razón de dependencia (ONU, 65+ / 15–64)

La razón de dependencia de vejez es el indicador habitual del envejecimiento. Se compara, **por cohorte $t$**, el $TBP_C$ con la razón de dependencia en el año $t+65$ (cuando la cohorte tiene 65, el momento que pondera el TBP) y en $t+40$ (mitad de la vida activa). Esperado: **signo negativo** (más dependencia ⇒ menor TBP).

**⚠ Circularidad mecánica.** $J$ (población 65+ en $t+65$) es el numerador de la razón de dependencia **en $t+65$** y el denominador de $TBP_C$: un signo negativo en $t+65$ está **garantizado en parte por construcción**. La comparación en $t+40$ evita esa coincidencia exacta de año, pero ambas series tienen **tendencia** (envejecimiento), de modo que una correlación de rangos entre series con tendencia es casi inevitable. Se agregan dos diagnósticos: (i) la correlación de la razón con $1/J$ solo (benchmark mecánico) y (ii) las **primeras diferencias** (cambios entre cohortes consecutivas). **No se reportan p-valores** (cohortes consecutivas no son independientes).

In [ ]:
dep = s["un_dependencia_65mas_15_64"]
# la columna de la ONU = pob 65+ / pob 15-64 (control de consistencia con las series de poblacion)
chk = (s.un_pob_65mas_miles / s.un_pob_15_64_miles).dropna()
assert np.allclose(dep.loc[chk.index], chk, atol=1e-3)

x = pd.DataFrame({"cohorte": c.index, "TBP_C": c.TBP_C_legal.values, "TBP_B": c.TBP_B_legal.values,
                  "inv_J": 1 / c.J.values, "NS65": (c.N_t * c.S65).values})
x["dep_t65"] = dep.reindex(c.anio_65.values).values
x["dep_t40"] = dep.reindex((c.index.values + 40)).values
x = x.dropna()

def sp(a, b, df=x): return df[a].corr(df[b], method="spearman")
def sp_dif(a, b, df=x): return df[a].diff().corr(df[b].diff(), method="spearman")
filas = []
for etiqueta, df_ in [("todas las cohortes 1950–2035", x), ("solo cohortes con N observado (≤2024)", x[x.cohorte <= 2024])]:
    for h in ["dep_t65", "dep_t40"]:
        filas.append(dict(muestra=etiqueta, razon_dependencia_en=h.replace("dep_t", "t+"), n=len(df_),
                          spearman_TBP_C=sp("TBP_C", h, df_), spearman_inv_J_benchmark_mecanico=sp("inv_J", h, df_),
                          spearman_NxS65_sin_J=sp("NS65", h, df_), spearman_TBP_B_sin_N_ni_J=sp("TBP_B", h, df_),
                          spearman_primeras_dif_TBP_C=sp_dif("TBP_C", h, df_)))
corr = pd.DataFrame(filas)
display(corr.round(3))
corr.to_csv(os.path.join(SAL, "s11_correlacion_dependencia.csv"), index=False)

r65 = sp("TBP_C", "dep_t65"); r40 = sp("TBP_C", "dep_t40")
print(f"Spearman(TBP_C, razón de dependencia en t+65) = {r65:+.3f}   <- inflado por circularidad (J en ambos)")
print(f"Spearman(TBP_C, razón de dependencia en t+40) = {r40:+.3f}")
print(f"Benchmark mecánico: Spearman(1/J, razón en t+65) = {sp('inv_J','dep_t65'):+.3f}")
assert r65 < 0 and r40 < 0
print("OK) signo negativo en ambos horizontes, como se esperaba.")
print("Lectura prudente: el signo es el esperado, pero no es una validación independiente (circularidad y tendencia común). "
      "La información adicional del TBP respecto de la razón de dependencia está en N, S65, E_r, τ, ρ y Ā, que la razón no incluye.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
for a, h, ttl in [(ax[0], "dep_t65", "razón de dependencia en t+65 (circular)"), (ax[1], "dep_t40", "razón de dependencia en t+40")]:
    sc = a.scatter(x[h], x.TBP_C, c=x.cohorte, cmap="viridis", s=18)
    a.set_xlabel(ttl); a.set_ylabel("TBP_C (Ā=30, años)"); a.grid(alpha=.3)
    a.set_title(f"Spearman = {sp('TBP_C', h):+.2f}", fontsize=10)
fig.colorbar(sc, ax=ax, label="cohorte")
plt.savefig(os.path.join(SAL, "s11_tbp_vs_dependencia.png"), dpi=130, bbox_inches="tight"); plt.show()

### 11.4 Qué se puede y qué no se puede concluir

* **Sí:** el signo y el orden de magnitud de cada canal (H1–H6) con cohortes reales; que la caída del $TBP_C$ 2000→2024 se descompone **exactamente** en natalidad, supervivencia a 65 y stock de mayores; que $\rho$, $\tau$ y $\bar A$ cambian el **nivel** y no la forma.
* **No:** probar causalidad ni significancia; afirmar que $\rho$ o $\tau$ sean constantes entre cohortes (hoy se usa un valor único de 2023); evaluar H4 sin $\bar A$ por cohorte (EPH, sección 8, sin datos reales aún).
* **Cohorte 2024 y siguientes:** $J$, $S_{65}$ y $E_r$ son **proyecciones ONU a 2089+**: la incertidumbre demográfica es grande y no está en el tornado salvo la ilustración de $J$ ±10%.
* Los ejemplos rotulados "ILUSTRATIVO" (ρ dependiente de la cohorte, $N$ y $J$ ±10%) **no son datos**.

In [ ]:
# ---- Controles finales (4a, 4b, 4c ya ejecutados arriba) y listado de salidas ----
assert os.path.isdir(SAL) and "procesados" not in os.path.abspath(SAL)
for f in sorted(os.listdir(SAL)):
    if f.startswith("s11_"): print("Escrito:", os.path.join(SAL, f))
print("\nControles de la sección 11: 4a linealidad OK | 4b descomposición suma ΔlnTBP OK | 4c parámetros citados OK")